In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# ניטור או ביטול עבודה

צפה ברשימת עומסי העבודה שלך ב[דף עומסי העבודה](https://quantum.cloud.ibm.com/workloads).

## צפייה בסטטוס עבודה
עבור ל[טבלת עומסי העבודה](https://quantum.cloud.ibm.com/workloads) שלך ובדוק תחת עמודת הסטטוס אם עבודה הושלמה או נכשלה.

## צפייה בשימוש הנותר
עבור ל[טבלת ה-Instances](https://quantum.cloud.ibm.com/instances) שלך ובחר את הלשונית המשויכת לתוכנית שברצונך לצפות בשימוש הנותר לה. הזמן הכולל שנוצל והזמן הכולל שנותר בתוכנית שלך מוצגים.

## צפייה במדדים על מספר העבודות ועומסי העבודה שהוגשו
עבור ל[דף Analytics](https://quantum.cloud.ibm.com/analytics) כדי לראות את המספר הכולל של העבודות שהוגשו, כמו גם ספירה של עומסי עבודה במצב batch ועומסי עבודה במצב session. שים לב שניתן לראות את דף Analytics רק עבור חשבונות שבבעלותך או שאתה מנהל.

## ניטור עבודה
השתמש ב-instance של העבודה כדי לבדוק את סטטוס העבודה או לאחזר את התוצאות על ידי קריאה לפקודה המתאימה:

|                               |                                                                                                                                                                                                              |
| ----------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| job.result()                  | ראה בתוצאות העבודה מיד לאחר השלמתה. תוצאות העבודה זמינות לאחר השלמת העבודה. לפיכך, job.result() היא קריאה חוסמת עד להשלמת העבודה.                               |
| job.job\_id()                 | החזר את המזהה הייחודי של אותה עבודה. אחזור תוצאות העבודה בשלב מאוחר יותר דורש את מזהה העבודה. לפיכך, מומלץ לשמור את המזהים של עבודות שייתכן שתרצה לאחזר מאוחר יותר. |
| job.status()                  | בדוק את סטטוס העבודה.                                                                                                                                                                                        |
| job = service.job(\<job\_id>) | אחזר עבודה שהגשת בעבר. קריאה זו דורשת את מזהה העבודה.                                                                                                                                      |

<span id="retrieve-later"></span>
## אחזור תוצאות עבודה בשלב מאוחר יותר
קרא ל-`service.job(\<job\_id>)` כדי לאחזר עבודה שהגשת בעבר. אם אין לך את מזהה העבודה, או אם ברצונך לאחזר מספר עבודות בבת אחת; כולל עבודות מ-QPUs (יחידות עיבוד קוונטי) שפרשו, קרא ל-`service.jobs()` עם פילטרים אופציונליים במקום. ראה [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs).

> **Note:** `service.jobs()` מחזיר גם עבודות שהורצו מחבילת `qiskit-ibm-provider` המיושנת. עבודות שהוגשו על ידי חבילת `qiskit-ibmq-provider` הישנה יותר (גם כן מיושנת) אינן זמינות עוד.

### דוגמה
דוגמה זו מחזירה את 10 עבודות ה-runtime האחרונות שהורצו על `my_backend`:

In [ ]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new Estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

## Next steps

<Admonition type="tip" title="Recommendations">
    - Try the [Grover's algorithm](/docs/tutorials/grovers-algorithm) tutorial.
    - Learn more about [Sampler execution spans](/docs/guides/sampler-input-output#execution-spans)
</Admonition>